In [2]:
paper = """
================================================================================
        EVALUATING MACHINE LEARNING TECHNIQUES FOR
              LOG-BASED ANOMALY DETECTION
================================================================================

Author  : [Your Name]
Program : BTech Computer Science
Dataset : HDFS_v1 (LogHub) — 575,061 sessions
Date    : 2024

================================================================================
ABSTRACT
================================================================================

This paper evaluates four unsupervised machine learning algorithms —
Isolation Forest, One-Class SVM, Local Outlier Factor (LOF), and DBSCAN —
for detecting anomalous behavior in Hadoop Distributed File System (HDFS)
logs. Using 575,061 log sessions with 19 engineered features derived from
raw system logs, we compare algorithm performance across five metrics:
Precision, Recall, F1 Score, ROC-AUC, and False Positive Rate.

Isolation Forest achieved the highest ROC-AUC (0.9609), while One-Class SVM
achieved the best F1 Score (0.6491) and Recall (0.7468). LOF underperformed
significantly (ROC-AUC = 0.5924), and DBSCAN demonstrated the lowest false
positive rate (0.0065) at the cost of high missed detections. Results suggest
Isolation Forest is most suitable for production security monitoring, while
One-Class SVM is preferred when maximizing anomaly detection rate.

================================================================================
1. INTRODUCTION
================================================================================

1.1 Motivation
--------------
Modern distributed systems generate millions of log entries daily. Manual
inspection of these logs is infeasible at scale. Automated anomaly detection
using machine learning offers a scalable alternative for security monitoring,
fault detection, and compliance auditing.

1.2 Problem Statement
---------------------
Given raw system log data from a distributed file system, can unsupervised
machine learning algorithms reliably identify anomalous sessions without
requiring labeled training data?

1.3 Research Questions
----------------------
RQ1: Which unsupervised ML algorithm achieves the best overall performance
     for log-based anomaly detection on the HDFS dataset?

RQ2: How does feature engineering quality affect model performance
     independently of algorithm choice?

RQ3: What is the tradeoff between false positive rate and recall across
     the four evaluated algorithms?

1.4 Contributions
-----------------
1. A systematic comparison of 4 unsupervised anomaly detection algorithms
   on real-world HDFS log data at scale (575k sessions)
2. A feature engineering pipeline producing 19 session-level features
   from raw unstructured log text
3. Empirical evidence that Isolation Forest outperforms density-based
   methods (LOF, DBSCAN) on high-dimensional log feature spaces

================================================================================
2. BACKGROUND & RELATED WORK
================================================================================

2.1 Log-Based Anomaly Detection
--------------------------------
System logs record runtime behavior of software components. Log-based
anomaly detection involves parsing unstructured log text, extracting
meaningful features, and identifying sessions that deviate from normal
operational patterns.

2.2 Algorithms Overview
------------------------
Isolation Forest (Liu et al., 2008):
  Isolates anomalies using random recursive partitioning. Anomalies,
  being few and different, require fewer splits to isolate. Time
  complexity O(n log n), making it scalable to large datasets.

One-Class SVM (Schölkopf et al., 1999):
  Learns a decision boundary enclosing normal data in a kernel-induced
  feature space. Points outside the boundary are classified as anomalies.
  Computationally expensive O(n²) — requires sampling at scale.

Local Outlier Factor (Breunig et al., 2000):
  Assigns anomaly scores based on local density deviation. Points in
  low-density regions relative to their neighbors are anomalies.
  Sensitive to high dimensionality and scale.

DBSCAN (Ester et al., 1996):
  Density-based clustering that labels unclustered points as noise/anomalies.
  Requires tuning of eps and min_samples hyperparameters.

================================================================================
3. METHODOLOGY
================================================================================

3.1 Dataset
-----------
Dataset  : HDFS_v1 from LogHub benchmark collection
Source   : Hadoop Distributed File System private cloud environment
Size     : 11,175,629 raw log lines → 575,061 unique sessions
Labels   : 558,223 Normal (97.07%) | 16,838 Anomaly (2.93%)
Labeling : Manual expert annotation based on block-level traces

3.2 Preprocessing Pipeline
---------------------------
Step 1: Raw log parsing using regex — extracted Date, Time, PID,
        Level, Component, Content from each log line
Step 2: Block ID extraction from log content using pattern blk_[-]?\\d+
Step 3: Session construction — grouped log lines by Block ID
Step 4: Label attachment — joined session groups with anomaly_label.csv

3.3 Feature Engineering
------------------------
19 features were engineered across 7 categories:

Category              Features
─────────────────────────────────────────────────────
Session Activity    : session_length
Log Level Counts    : count_INFO, count_WARN,
                      count_ERROR, count_DEBUG
Component Activity  : unique_components,
                      comp_PacketResponder,
                      comp_DataXceiver,
                      comp_FSNamesystem,
                      comp_FSDataset,
                      comp_DataBlockScanner
Temporal            : duration
Keyword Signals     : kw_exception, kw_failed,
                      kw_error, kw_timeout,
                      kw_replicating, kw_served
Process Behavior    : unique_pids

Feature Importance Analysis (Mean Absolute Difference):
  1. comp_PacketResponder  — 3.547 (highest)
  2. kw_error              — 3.118
  3. kw_replicating        — 1.854
  4. count_INFO            — 0.606
  5. unique_pids           — 0.588

3.4 Experimental Setup
-----------------------
- All models trained in unsupervised mode (no labeled training data)
- StandardScaler applied to normalize all features
- Contamination parameter set to 0.0293 (known anomaly rate)
- LOF and DBSCAN applied with 30k/20k subsampling due to O(n²) complexity
- Evaluation performed on full dataset (575,061 sessions)

================================================================================
4. RESULTS
================================================================================

4.1 Quantitative Comparison
-----------------------------

Model                Precision  Recall   F1     ROC-AUC   FPR
─────────────────────────────────────────────────────────────
Isolation Forest      0.6219    0.6223  0.6221   0.9609   0.0114
One-Class SVM         0.5740    0.7468  0.6491   0.9246   0.0167
Local Outlier Factor  0.1455    0.1530  0.1492   0.5924   0.0271
DBSCAN                0.6072    0.3346  0.4315   0.5000   0.0065

4.2 Key Observations
---------------------
O1: Isolation Forest achieves the highest ROC-AUC (0.9609), indicating
    superior global discrimination between normal and anomalous sessions.

O2: One-Class SVM achieves the best F1 (0.6491) and Recall (0.7468),
    detecting 74.68% of all true anomalies — critical for security use cases.

O3: LOF's near-random ROC-AUC (0.5924) demonstrates that local density
    methods fail on high-dimensional session feature spaces.

O4: DBSCAN's near-zero FPR (0.0065) suggests ultra-conservative flagging —
    useful when false alarms are operationally costly.

O5: Anomaly score distribution plots confirm Isolation Forest produces
    the clearest score separation between normal and anomalous sessions.

O6: PCA visualization reveals anomalies are not uniformly distributed —
    a subset forms an isolated cluster while others overlap with normal
    data, suggesting multiple anomaly subtypes.

================================================================================
5. DISCUSSION
================================================================================

5.1 Algorithm Suitability
--------------------------
For production security monitoring where catching all threats matters,
One-Class SVM's superior recall (0.7468) is preferable despite slightly
higher false alarm rate. For general-purpose anomaly scoring with minimal
configuration, Isolation Forest offers the best balance.

5.2 Feature Engineering Impact
--------------------------------
The dominance of comp_PacketResponder and kw_error as discriminative
features aligns with HDFS domain knowledge — block replication failures
produce characteristic log patterns detectable through session-level
feature aggregation.

5.3 Scalability Considerations
--------------------------------
LOF and DBSCAN required subsampling (30k/20k of 575k sessions) due to
quadratic time complexity. This directly impacted their evaluation quality.
Isolation Forest and One-Class SVM scaled more effectively, with Isolation
Forest being the most computationally efficient.

================================================================================
6. CONCLUSION
================================================================================

This study evaluated four unsupervised ML algorithms for log-based anomaly
detection on 575,061 HDFS sessions. Isolation Forest emerged as the best
overall performer (ROC-AUC=0.9609), while One-Class SVM was superior for
maximizing anomaly recall. Density-based methods (LOF, DBSCAN) underperformed
on this high-dimensional feature space.

Future work could explore: (1) deep learning approaches such as LSTM autoencoders
for sequence-aware log anomaly detection, (2) online/streaming anomaly detection
for real-time SOC deployment, and (3) ensemble methods combining Isolation Forest
with One-Class SVM for improved precision-recall balance.

================================================================================
REFERENCES
================================================================================

[1] Liu, F.T., Ting, K.M., Zhou, Z.H. (2008). Isolation Forest. ICDM 2008.
[2] Schölkopf, B. et al. (1999). Support Vector Method for Novelty Detection.
    NeurIPS 1999.
[3] Breunig, M.M. et al. (2000). LOF: Identifying Density-Based Local Outliers.
    SIGMOD 2000.
[4] Ester, M. et al. (1996). A Density-Based Algorithm for Discovering Clusters.
    KDD 1996.
[5] He, S. et al. (2020). Loghub: A Large Collection of System Log Datasets.
    IEEE ISSRE 2020.

================================================================================
"""

print(paper)

# Save to file
with open("reports/research_paper_draft.txt", "w", encoding="utf-8") as f:
    f.write(paper)

print(" Paper draft saved to reports/research_paper_draft.txt")


        EVALUATING MACHINE LEARNING TECHNIQUES FOR
              LOG-BASED ANOMALY DETECTION

Author  : Anshika Pandey
Program : BTech Computer Science
Dataset : HDFS_v1 (LogHub) — 575,061 sessions
Date    : 2024

ABSTRACT

This paper evaluates four unsupervised machine learning algorithms —
Isolation Forest, One-Class SVM, Local Outlier Factor (LOF), and DBSCAN —
for detecting anomalous behavior in Hadoop Distributed File System (HDFS)
logs. Using 575,061 log sessions with 19 engineered features derived from
raw system logs, we compare algorithm performance across five metrics:
Precision, Recall, F1 Score, ROC-AUC, and False Positive Rate.

Isolation Forest achieved the highest ROC-AUC (0.9609), while One-Class SVM
achieved the best F1 Score (0.6491) and Recall (0.7468). LOF underperformed
significantly (ROC-AUC = 0.5924), and DBSCAN demonstrated the lowest false
positive rate (0.0065) at the cost of high missed detections. Results suggest
Isolation Forest is most suitable for produc